In [1]:
#Importando Bibliotecas 

import pandas as pd
import datetime
import re
import datetime
from datetime import time

from pyspark.sql.functions import current_timestamp

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 3, Finished, Available, Finished)

In [2]:
#Ler a aba dSuporte da URL

url = (
    "https://teste7857118138.blob.core.windows.net/azureml/"
    "Base%20SAC.xlsx?"
    "sp=r&st=2026-02-07T16:15:43Z"
    "&se=2026-02-08T00:30:43Z"
    "&spr=https"
    "&sv=2024-11-04"
    "&sr=b"
    "&sig=H3YESyvwvLZ8TTgFIH%2BrWFRuuhbCN%2BgC2TqTQAsoT3E%3D"
)

df = pd.read_excel(
    url,
    sheet_name="dSuporte"
)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 4, Finished, Available, Finished)

In [3]:
#Converter campos datetime.time
for col in df.columns:
    df[col] = df[col].apply(
        lambda x: x.strftime('%H:%M:%S')
        if isinstance(x, time)
        else x
    )

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 5, Finished, Available, Finished)

In [4]:
#Normalizar nomes das colunas

def normalize_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^\w]", "_", col)
    col = re.sub(r"_+", "_", col)
    return col

df.columns = [normalize_col(c) for c in df.columns]

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 6, Finished, Available, Finished)

In [5]:
#Converter Pandas → Spark

df_spark = spark.createDataFrame(df)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 7, Finished, Available, Finished)

In [9]:
#Adicionar coluna de auditoria

df_spark = df_spark.withColumn(
    "data_atualizacao",
    current_timestamp()
)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 11, Finished, Available, Finished)

In [10]:
#Criar tabela se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS dbo")

tabela = "dbo.dim_suporte"

if not spark.catalog.tableExists(tabela):
    (
        df_spark
        .limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela)
    )

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 12, Finished, Available, Finished)

In [11]:
#Ler dados existentes
df_bronze = spark.table(tabela)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 13, Finished, Available, Finished)

In [12]:
#Identificar novos registros (incremental)

df_incremental = (
    df_spark.alias("n")
    .join(
        df_bronze.alias("b"),
        on=["id_suporte"],
        how="left_anti"
    )
)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 14, Finished, Available, Finished)

In [13]:
#Gravar apenas registros novos

(
    df_incremental.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(tabela)
)

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 15, Finished, Available, Finished)

In [14]:
#Validar os dados na tabela Delta
#display(spark.sql("SELECT * FROM dbo.dim_suporte LIMIT 1000"))

StatementMeta(, fb5adaee-5713-4a91-90a2-43724928c38d, 16, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a326714d-8726-4e26-a133-85fc628bb519)